# CartPole neuroevolution 🎢
Balance a pole on a moving cart, the "hello world" of control and RL, but without gradients and without any RL library. The physics is 15 lines of NumPy (the classic Barto, Sutton & Anderson 1983 equations). The "brain" is just **4 weights**; we find them by **evolution**: mutate, keep what balances longer.

Background: [[RL Basics and MDPs]], [[Q-Learning and Policy Gradients]] · overview: [[Fun Projects]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import math, numpy as np, matplotlib.pyplot as plt
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
G, M_CART, M_POLE, L, F_MAG, TAU = 9.8, 1.0, 0.1, 0.5, 10.0, 0.02          # the classic CartPole constants
def step(s, action):
    x, x_dot, th, th_dot = s; force = F_MAG if action == 1 else -F_MAG
    total = M_CART + M_POLE; tmp = (force + M_POLE * L * th_dot ** 2 * math.sin(th)) / total
    th_acc = (G * math.sin(th) - math.cos(th) * tmp) / (L * (4 / 3 - M_POLE * math.cos(th) ** 2 / total))
    x_acc = tmp - M_POLE * L * th_acc * math.cos(th) / total
    s = (x + TAU * x_dot, x_dot + TAU * x_acc, th + TAU * th_dot, th_dot + TAU * th_acc)
    done = abs(s[0]) > 2.4 or abs(s[2]) > 12 * math.pi / 180
    return s, done
def episode_return(w, max_steps=500, seed=0, trace=False):
    r = np.random.default_rng(seed); s = tuple(r.uniform(-0.05, 0.05, 4)); states = [s]
    for t in range(max_steps):
        s, done = step(s, act(w, s)); states.append(s)
        if done: return (t + 1, states) if trace else t + 1
    return (max_steps, states) if trace else max_steps

## 1. A linear policy
Push right (action 1) if $w\cdot s > 0$, else left (0). State = (cart position, cart velocity, pole angle, pole angular velocity).

In [ ]:
def act(w, state):
    # TODO 1: 1 if w·state > 0 else 0
    raise NotImplementedError  # TODO 1

## 2. (1+λ) evolution strategy
Start from random weights. Each generation, try `pop` mutants `best + N(0, σ²)` and keep any that balances longer than the current best. Stop improving once you hit 500 steps. Return `(best_weights, history_of_best_scores)` with one entry per generation plus the start.

In [ ]:
def evolve(generations=30, pop=10, sigma=0.5, seed=0):
    # TODO 2: hill climbing in weight space on episode_return
    raise NotImplementedError  # TODO 2

In [ ]:
check("policy", lambda: act(np.array([0, 0, 1.0, 0]), (0, 0, 0.1, 0)) == 1 and act(np.array([0, 0, 1.0, 0]), (0, 0, -0.1, 0)) == 0, "Pole leaning right (+angle) with w = (0,0,1,0) → push right.")
if ready("policy"):
    print("random weights balance for:", [episode_return(np.random.default_rng(s).normal(0, 1, 4)) for s in range(8)], "steps")
    try:
        w, hist = evolve(); print("best weights:", np.round(w, 2), "| balances", hist[-1], "steps")
        print("on 10 unseen starting positions:", [episode_return(w, seed=s) for s in range(1, 11)])
        n, states = episode_return(w, trace=True); S_ = np.array(states)
        fig, ax = plt.subplots(1, 2, figsize=(11, 3)); ax[0].plot(hist, "o-"); ax[0].set_title("best steps balanced per generation")
        ax[1].plot(np.degrees(S_[:, 2]), label="pole angle (°)"); ax[1].plot(S_[:, 0], label="cart position (m)"); ax[1].legend(); ax[1].set_title("a 10-second balance"); plt.show()
        check("balances", lambda: hist[-1] == 500 and np.mean([episode_return(w, seed=s) for s in range(1, 11)]) > 400,
              "Evolution should find weights that balance the full 500 steps, and mostly from unseen starts too.")
    except NotImplementedError:
        print("❌ balances: not implemented yet – fill in the TODO above.")

<details><summary>▶ Solution</summary>

```python
def act(w, state):
    return int(np.dot(w, state) > 0)
```

```python
def evolve(generations=30, pop=10, sigma=0.5, seed=0):
    rng = np.random.default_rng(seed)
    best = rng.normal(0, 1, 4); best_score = episode_return(best)
    history = [best_score]
    for _ in range(generations):
        if best_score >= 500:
            history.append(best_score); continue
        for _ in range(pop):
            cand = best + rng.normal(0, sigma, 4)
            s = episode_return(cand)
            if s > best_score:
                best, best_score = cand, s
        history.append(best_score)
    return best, history
```
</details>

## Stretch goals
- Evaluate each candidate on 5 random starts instead of 1 (more robust, 5× slower).
- Replace the linear policy with a tiny neural net (4 → 8 → 1) and evolve its 49 weights.
- Solve it with REINFORCE ([[Q-Learning and Policy Gradients]]) and compare episodes needed.